In [35]:

import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (accuracy_score, classification_report,
                                     confusion_matrix, ConfusionMatrixDisplay)

OUT = "D:/PITP/files (2)/OUTPUT"
os.makedirs(OUT, exist_ok=True)

PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860"]
sns.set_theme(style="whitegrid", palette=PALETTE, font_scale=1.1)


print("=" * 60)
print("STEP 1: Loading Dataset")
print("=" * 60)

df = pd.read_csv("StudentsPerformance.csv")
print(f"\nShape   : {df.shape[0]} rows × {df.shape[1]} columns")
print("\nFirst 5 rows:")
print(df.head().to_string())
print("\nColumn data types:")
print(df.dtypes.to_string())


print("\n" + "=" * 60)
print("STEP 2: Data Cleaning & Missing-Value Handling")
print("=" * 60)

print("\nMissing values per column:")
print(df.isnull().sum().to_string())

print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()

print("\nBasic statistics (numeric columns):")
print(df.describe().to_string())


print("\n" + "=" * 60)
print("STEP 3: Feature Engineering & Label Encoding")
print("=" * 60)

df["average_score"] = (df["math_score"] + df["reading_score"] + df["writing_score"]) / 3

def classify_performance(avg):
    if avg >= 70:
        return "High"
    elif avg >= 50:
        return "Medium"
    else:
        return "Low"

df["performance_level"] = df["average_score"].apply(classify_performance)
print("\nPerformance level distribution:")
print(df["performance_level"].value_counts().to_string())

label_encoders = {}
cat_cols = ["gender", "parental_level_of_education", "lunch", "test_preparation_course"]
df_encoded = df.copy()

for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

target_le = LabelEncoder()
df_encoded["performance_level_encoded"] = target_le.fit_transform(df["performance_level"])
print(f"\nTarget classes: {dict(zip(target_le.classes_, target_le.transform(target_le.classes_)))}")


print("\n" + "=" * 60)
print("STEP 4: Exploratory Data Analysis")
print("=" * 60)


STEP 1: Loading Dataset

Shape   : 1000 rows × 7 columns

First 5 rows:
   gender parental_level_of_education         lunch test_preparation_course  math_score  reading_score  writing_score
0    male             master's degree      standard               completed          85             72             89
1  female            some high school  free/reduced               completed          80             61             43
2    male            some high school  free/reduced                    none          52             82             62
3    male            some high school      standard                    none          66             74             52
4    male           bachelor's degree  free/reduced                    none          72             53             82

Column data types:
gender                           str
parental_level_of_education      str
lunch                            str
test_preparation_course          str
math_score                     int64
reading_score  

In [36]:

# ── Figure 1: Score Distributions ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Score Distributions", fontsize=16, fontweight="bold")
for ax, col, color in zip(axes,
                           ["math_score", "reading_score", "writing_score"],
                           PALETTE[:3]):
    sns.histplot(df[col], kde=True, ax=ax, color=color, bins=25)
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Score")
    ax.axvline(df[col].mean(), color="red", linestyle="--", linewidth=1.5,
               label=f"Mean: {df[col].mean():.1f}")
    ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/01_score_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → 01_score_distributions.png")


  Saved → 01_score_distributions.png


In [37]:

# ── Figure 2: Performance Level Count ───────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
order = ["High", "Medium", "Low"]
counts = df["performance_level"].value_counts()[order]
bars = ax.bar(order, counts, color=PALETTE[:3], width=0.5, edgecolor="white", linewidth=1.2)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 8,
            str(count), ha="center", va="bottom", fontweight="bold")
ax.set_title("Performance Level Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Performance Level")
ax.set_ylabel("Number of Students")
ax.set_ylim(0, counts.max() + 60)
plt.tight_layout()
plt.savefig(f"{OUT}/02_performance_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → 02_performance_distribution.png")


  Saved → 02_performance_distribution.png


In [38]:

# ── Figure 3: Test Prep vs Scores ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Test Preparation Course vs Scores", fontsize=14, fontweight="bold")
for ax, col in zip(axes, ["math_score", "reading_score", "writing_score"]):
    sns.boxplot(data=df, x="test_preparation_course", y=col, ax=ax,
                palette={"completed": PALETTE[0], "none": PALETTE[1]})
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Test Preparation")
plt.tight_layout()
plt.savefig(f"{OUT}/03_test_prep_vs_scores.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → 03_test_prep_vs_scores.png")


  Saved → 03_test_prep_vs_scores.png


In [39]:

# ── Figure 4: Gender vs Scores ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Gender vs Scores", fontsize=14, fontweight="bold")
for ax, col in zip(axes, ["math_score", "reading_score", "writing_score"]):
    sns.violinplot(data=df, x="gender", y=col, ax=ax,
                   palette={"male": PALETTE[0], "female": PALETTE[1]},
                   inner="quartile")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Gender")
plt.tight_layout()
plt.savefig(f"{OUT}/04_gender_vs_scores.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → 04_gender_vs_scores.png")



  Saved → 04_gender_vs_scores.png


In [40]:
# ── Figure 5: Parental Education vs Average Score ────────────
edu_order = ["some high school", "high school", "some college",
             "associate's degree", "bachelor's degree", "master's degree"]
edu_avg = df.groupby("parental_level_of_education")["average_score"].mean().reindex(edu_order)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(edu_order, edu_avg, color=PALETTE[0], edgecolor="white")
for bar, val in zip(bars, edu_avg):
    ax.text(val + 0.4, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}", va="center", fontweight="bold")
ax.set_title("Parental Education vs Average Score", fontsize=14, fontweight="bold")
ax.set_xlabel("Average Score")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.savefig(f"{OUT}/05_parental_edu_vs_score.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → 05_parental_edu_vs_score.png")


  Saved → 05_parental_edu_vs_score.png


In [41]:

# ── Figure 6: Correlation Heatmap ────────────────────────────
numeric_cols = ["math_score", "reading_score", "writing_score", "average_score"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", mask=mask,
            ax=ax, linewidths=0.5, square=True,
            cbar_kws={"shrink": 0.8})
ax.set_title("Score Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/06_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → 06_correlation_heatmap.png")


  Saved → 06_correlation_heatmap.png


In [42]:

# ── Figure 7: Lunch Type vs Performance ──────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ct = pd.crosstab(df["lunch"], df["performance_level"])[["High", "Medium", "Low"]]
ct.plot(kind="bar", ax=ax, color=PALETTE[:3], edgecolor="white", width=0.6)
ax.set_title("Lunch Type vs Performance Level", fontsize=14, fontweight="bold")
ax.set_xlabel("Lunch Type")
ax.set_ylabel("Number of Students")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Performance")
plt.tight_layout()
plt.savefig(f"{OUT}/07_lunch_vs_performance.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → 07_lunch_vs_performance.png")


  Saved → 07_lunch_vs_performance.png


In [43]:

#  STEP 5 — Preprocessing & Splitting
print("\n" + "=" * 60)
print("STEP 5: Preprocessing & Splitting")
print("=" * 60)

feature_cols = cat_cols + ["math_score", "reading_score", "writing_score"]
X = df_encoded[feature_cols]
y = df_encoded["performance_level_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\nTraining set : {X_train.shape[0]} samples")
print(f"Testing set  : {X_test.shape[0]} samples")
print(f"Features     : {feature_cols}")



STEP 5: Preprocessing & Splitting

Training set : 800 samples
Testing set  : 200 samples
Features     : ['gender', 'parental_level_of_education', 'lunch', 'test_preparation_course', 'math_score', 'reading_score', 'writing_score']


In [44]:

#  STEP 6 — Model Building (Random Forest)
print("\n" + "=" * 60)
print("STEP 6: Model Building — Random Forest Classifier")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
print("\nModel trained successfully!")
print(f"  n_estimators : {rf_model.n_estimators}")
print(f"  max_depth    : {rf_model.max_depth}")



STEP 6: Model Building — Random Forest Classifier

Model trained successfully!
  n_estimators : 200
  max_depth    : None


In [45]:

#  STEP 7 — Model Evaluation
print("\n" + "=" * 60)
print("STEP 7: Model Evaluation")
print("=" * 60)

y_pred  = rf_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
class_names = target_le.classes_  

print(f"\nTest Accuracy : {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                             target_names=class_names))



STEP 7: Model Evaluation

Test Accuracy : 91.50%

Classification Report:
              precision    recall  f1-score   support

        High       0.95      0.96      0.95       116
         Low       1.00      0.14      0.25         7
      Medium       0.87      0.92      0.89        77

    accuracy                           0.92       200
   macro avg       0.94      0.67      0.70       200
weighted avg       0.92      0.92      0.91       200



In [46]:

# ── Figure 8: Confusion Matrix ───────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=False, cmap="Blues",
          values_format="d")
ax.set_title(f"Confusion Matrix  (Accuracy: {accuracy*100:.2f}%)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/08_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  Saved → 08_confusion_matrix.png")



  Saved → 08_confusion_matrix.png


In [47]:

#  STEP 8 — Feature Importance
print("\n" + "=" * 60)
print("STEP 8: Feature Importance")
print("=" * 60)

importances = rf_model.feature_importances_
feat_df = pd.DataFrame({
    "Feature"   : feature_cols,
    "Importance": importances
}).sort_values("Importance", ascending=True)

print("\nFeature importances (sorted):")
for _, row in feat_df.iterrows():
    bar = "█" * int(row["Importance"] * 50)
    print(f"  {row['Feature']:40s} {row['Importance']:.4f}  {bar}")

fig, ax = plt.subplots(figsize=(9, 6))
colors = [PALETTE[0] if v > 0.15 else PALETTE[2] if v > 0.05 else PALETTE[1]
          for v in feat_df["Importance"]]
bars = ax.barh(feat_df["Feature"], feat_df["Importance"],
               color=colors, edgecolor="white", height=0.6)
for bar, val in zip(bars, feat_df["Importance"]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=9)
ax.set_title("Random Forest — Feature Importance", fontsize=14, fontweight="bold")
ax.set_xlabel("Importance Score")
ax.set_xlim(0, feat_df["Importance"].max() + 0.06)
plt.tight_layout()
plt.savefig(f"{OUT}/09_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  Saved → 09_feature_importance.png")



STEP 8: Feature Importance

Feature importances (sorted):
  gender                                   0.0130  
  lunch                                    0.0226  █
  test_preparation_course                  0.0279  █
  parental_level_of_education              0.0485  ██
  writing_score                            0.2676  █████████████
  math_score                               0.3030  ███████████████
  reading_score                            0.3173  ███████████████

  Saved → 09_feature_importance.png


In [48]:

#  STEP 9 — Predict for New Students
print("\n" + "=" * 60)
print("STEP 9: Predicting Performance for New Students")
print("=" * 60)

new_students = pd.DataFrame({
    "gender"                    : ["female", "male",   "female"],
    "parental_level_of_education": ["bachelor's degree", "some high school", "master's degree"],
    "lunch"                     : ["standard", "free/reduced", "standard"],
    "test_preparation_course"   : ["completed", "none", "completed"],
    "math_score"                : [85,          45,      90],
    "reading_score"             : [88,          50,      92],
    "writing_score"             : [90,          48,      95],
})

new_encoded = new_students.copy()
for col in cat_cols:
    new_encoded[col] = label_encoders[col].transform(new_students[col])

new_scaled = scaler.transform(new_encoded[feature_cols])
predictions      = rf_model.predict(new_scaled)
probabilities    = rf_model.predict_proba(new_scaled)
predicted_labels = target_le.inverse_transform(predictions)

print("\nPrediction Results:")
print("-" * 70)
for i, (label, probs) in enumerate(zip(predicted_labels, probabilities)):
    avg = (new_students.loc[i, "math_score"] +
           new_students.loc[i, "reading_score"] +
           new_students.loc[i, "writing_score"]) / 3
    conf = max(probs) * 100
    print(f"  Student {i+1}  |  Avg Score: {avg:.1f}  |  "
          f"Predicted: {label:6s}  |  Confidence: {conf:.1f}%")
    for cls, p in zip(target_le.classes_, probs):
        print(f"             {cls:8s}: {p*100:5.1f}%")
    print()

# ── Figure 9: New-Student Prediction Bar Chart ───────────────
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(predicted_labels))
width = 0.25
for j, cls in enumerate(target_le.classes_):
    ax.bar(x + j * width,
           [probabilities[i][j] * 100 for i in range(len(x))],
           width, label=cls, color=PALETTE[j], edgecolor="white")
ax.set_xticks(x + width)
ax.set_xticklabels([f"Student {i+1}" for i in range(len(x))])
ax.set_ylabel("Probability (%)")
ax.set_title("Prediction Probabilities for New Students",
             fontsize=13, fontweight="bold")
ax.legend(title="Performance Level")
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig(f"{OUT}/10_new_student_predictions.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → 10_new_student_predictions.png")



STEP 9: Predicting Performance for New Students

Prediction Results:
----------------------------------------------------------------------
  Student 1  |  Avg Score: 87.7  |  Predicted: High    |  Confidence: 100.0%
             High    : 100.0%
             Low     :   0.0%
             Medium  :   0.0%

  Student 2  |  Avg Score: 47.7  |  Predicted: Medium  |  Confidence: 56.9%
             High    :   0.1%
             Low     :  43.0%
             Medium  :  56.9%

  Student 3  |  Avg Score: 92.3  |  Predicted: High    |  Confidence: 99.5%
             High    :  99.5%
             Low     :   0.0%
             Medium  :   0.5%

  Saved → 10_new_student_predictions.png


In [49]:

#  SUMMARY DASHBOARD  (single combined figure)
print("\n" + "=" * 60)
print("Generating Summary Dashboard …")
print("=" * 60)

fig = plt.figure(figsize=(18, 12))
fig.suptitle("Student Performance Prediction — Project Summary Dashboard",
             fontsize=17, fontweight="bold", y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# (A) Performance distribution pie
ax_pie = fig.add_subplot(gs[0, 0])
perf_counts = df["performance_level"].value_counts()[["High", "Medium", "Low"]]
ax_pie.pie(perf_counts, labels=perf_counts.index,
           colors=PALETTE[:3], autopct="%1.1f%%",
           startangle=140, wedgeprops=dict(edgecolor="white", linewidth=1.5))
ax_pie.set_title("Performance Level Share", fontweight="bold")

# (B) Test Prep effect on math
ax_tp = fig.add_subplot(gs[0, 1])
sns.boxplot(data=df, x="test_preparation_course", y="math_score",
            ax=ax_tp, palette={"completed": PALETTE[0], "none": PALETTE[1]},
            width=0.5)
ax_tp.set_title("Test Prep vs Math Score", fontweight="bold")
ax_tp.set_xlabel("Test Preparation")

# (C) Parental edu vs avg score
ax_edu = fig.add_subplot(gs[0, 2])
edu_avg_plot = df.groupby("parental_level_of_education")["average_score"].mean().reindex(edu_order)
short_labels = ["Some HS", "HS", "Some Coll.", "Assoc.", "Bach.", "Master's"]
ax_edu.bar(short_labels, edu_avg_plot, color=PALETTE[0], edgecolor="white")
ax_edu.set_title("Parental Education vs Avg Score", fontweight="bold")
ax_edu.set_xticklabels(short_labels, rotation=30, ha="right")
ax_edu.set_ylabel("Average Score")
ax_edu.set_ylim(50, 90)

# (D) Confusion matrix mini
ax_cm = fig.add_subplot(gs[1, 0])
ConfusionMatrixDisplay(confusion_matrix=cm,
                       display_labels=class_names).plot(ax=ax_cm, colorbar=False,
                                                        cmap="Blues", values_format="d")
ax_cm.set_title(f"Confusion Matrix (Acc: {accuracy*100:.1f}%)", fontweight="bold")

# (E) Feature importance (top 5)
ax_fi = fig.add_subplot(gs[1, 1])
top5 = feat_df.tail(5)
ax_fi.barh(top5["Feature"], top5["Importance"],
           color=PALETTE[0], edgecolor="white")
ax_fi.set_title("Top-5 Feature Importances", fontweight="bold")
ax_fi.set_xlabel("Importance")

# (F) New student prediction
ax_pred = fig.add_subplot(gs[1, 2])
x2 = np.arange(3)
w2 = 0.25
for j, cls in enumerate(target_le.classes_):
    ax_pred.bar(x2 + j * w2,
                [probabilities[i][j] * 100 for i in range(3)],
                w2, label=cls, color=PALETTE[j], edgecolor="white")
ax_pred.set_xticks(x2 + w2)
ax_pred.set_xticklabels(["Student 1", "Student 2", "Student 3"])
ax_pred.set_ylabel("Probability (%)")
ax_pred.set_title("New Student Predictions", fontweight="bold")
ax_pred.legend(fontsize=8)
ax_pred.set_ylim(0, 115)

plt.savefig(f"{OUT}/00_dashboard.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → 00_dashboard.png")

print("\n" + "=" * 60)
print("PROJECT COMPLETE — all outputs saved to:", OUT)
print("=" * 60)
print(f"\nFinal Model Accuracy: {accuracy * 100:.2f}%")



Generating Summary Dashboard …
  Saved → 00_dashboard.png

PROJECT COMPLETE — all outputs saved to: D:/PITP/files (2)/OUTPUT

Final Model Accuracy: 91.50%
